# backward-func-lookup — worked example 3: Dispatch back fns for one two-parent node

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-func-lookup`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a reverse pass, the lookup is queried once per parent edge of a node. A node produced by a binary op like `t.multiply` has two tensor parents, so `get_back_func` is called twice — once with `argnum=0`, once with `argnum=1` — and each returns the correct partial-derivative back fn for that input slot.

## Worked solution

**Step 1 — register both argnums.** For `z = x * y`, the gradient w.r.t. `x` is `grad_out * y` (argnum 0) and w.r.t. `y` is `grad_out * x` (argnum 1). We register each under its own key.

**Step 2 — set up the forward value.** We pick concrete `x`, `y` and compute `out = x * y` so the back fns have the realistic signature `(grad_out, out, x, y)`.

**Step 3 — dispatch per edge.** Looping `for argnum in (0, 1)`, we call `BFL.get_back_func(t.multiply, argnum)` and apply it to the incoming `grad_out`. This is exactly the per-parent dispatch the real reverse pass performs.

**Step 4 — verify against autograd.** We recompute the same product with `requires_grad=True` tensors and `.backward()`. The grad the lookup-dispatched back fns produce must equal `x.grad` and `y.grad`. Matching confirms the right back fn landed in the right argnum slot — the whole point of keying by `(fwd, argnum)`.

In [ ]:
t.manual_seed(0)

class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn
    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={argnum}).')
        return self.back_funcs[key]

def mul_back0(grad_out, out, x, y):
    return grad_out * y
def mul_back1(grad_out, out, x, y):
    return grad_out * x

BFL = BackwardFuncLookup()
BFL.add_back_func(t.multiply, 0, mul_back0)
BFL.add_back_func(t.multiply, 1, mul_back1)

x = t.tensor([2.0, 3.0, 4.0])
y = t.tensor([5.0, 6.0, 7.0])
out = x * y
grad_out = t.ones_like(out)

grads = {}
for argnum in (0, 1):
    back_fn = BFL.get_back_func(t.multiply, argnum)
    grads[argnum] = back_fn(grad_out, out, x, y)

print('dx:', grads[0].tolist())
print('dy:', grads[1].tolist())